# TCL Phase 4a Runs (Qwen2.5-7B)

Phase 3's inference, unchanged, on Qwen2.5-7B-Instruct in float16 split across two T4s (`docs/phase4/TCL-Phase4a-Protocol.md`). Tasks:

- `pool`: TriviaQA, SQuAD, NQ-Open (Phase 1 question sets), semantic entropy on test splits.
- `popqa`: the first `N_POPQA` PopQA questions (1,500 = batch 1), semantic entropy.
- `grounding`: the four MRQA grounding sets with the passage and the abstention line, then grounding labels with the chosen labeler (`runs/phase4a/grounding_labeler_selection.json`).
- `relabel`: the Phase 3 grounding answers (Qwen 0.5B, Phi) relabeled with the chosen labeler (ungated check).

Each session writes one zip to `/kaggle/working/`. Re-running resumes finished questions.


## Settings

- Accelerator: **GPU T4 x2**. Internet: **on**.
- Smoke test: `LIMIT = 50`, `ONLY_SPLITS = "test"`. Full runs: `None` and `""`.


In [ ]:
# Pin the exact code and data version used for this run.
REPO = "https://github.com/awabmoha/truth-calibration-layer.git"
COMMIT = "5b40b300e1ebe051dfe0daec455b87bfa4ddfa33"

MODEL = "Qwen/Qwen2.5-7B-Instruct"
MODEL_TAG = "qwen7b"
TASKS = ["pool", "popqa", "grounding"]   # any of: pool, popqa, grounding, relabel
POOL_DATASETS = ["triviaqa", "squad", "nq_open"]
GROUNDING_DATASETS = ["squad", "nq", "hotpotqa", "relation_extraction"]
N_POPQA = 1500      # 3000 / 4500 only if the size rule needs it
LIMIT = None        # questions per dataset; 50 for the smoke test
ONLY_SPLITS = ""    # "test" for the smoke test (exercises semantic entropy)
TAG = "run"         # name of the output zip
print(MODEL, TASKS, LIMIT, ONLY_SPLITS)


In [ ]:
import json, os, shutil, subprocess
if not os.path.exists("truth-calibration-layer"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("truth-calibration-layer")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
os.chdir("tcl_experiments")
LABELER = json.load(open("runs/phase4a/grounding_labeler_selection.json"))["chosen"]
print("grounding labeler:", LABELER)


In [ ]:
!python scripts/check_runtime.py
!nvidia-smi --query-gpu=name,memory.total --format=csv


## Run


In [ ]:
OUT = "/kaggle/working/phase4a"
BASE = ["python", "scripts/run_inference_v2.py", "--model", MODEL, "--model-tag", MODEL_TAG,
        "--torch-dtype", "float16", "--device-map", "auto", "--resume"]

def run(cmd):
    if LIMIT:
        cmd = cmd + ["--limit", str(LIMIT)]
    if ONLY_SPLITS:
        cmd = cmd + ["--only-splits", ONLY_SPLITS]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

if "pool" in TASKS:
    for ds in POOL_DATASETS:
        run(BASE + ["--dataset", ds, "--questions", f"data/phase1_kaggle/{ds}.csv",
                    "--out-dir", f"{OUT}/phase1_kaggle/{MODEL_TAG}/{ds}", "--se-splits", "test"])

if "popqa" in TASKS:
    cmd = BASE + ["--dataset", "popqa", "--questions", "data/phase3/popqa.csv",
                  "--out-dir", f"{OUT}/phase3_popqa/{MODEL_TAG}/popqa", "--se-splits", "test"]
    if not LIMIT:
        cmd += ["--limit", str(N_POPQA)]
    run(cmd)

if "grounding" in TASKS:
    for ds in GROUNDING_DATASETS:
        out_dir = f"{OUT}/phase3_grounding/{MODEL_TAG}/{ds}"
        run(BASE + ["--dataset", ds, "--questions", f"data/phase3_grounding/{ds}.csv", "--out-dir", out_dir,
                    "--se-splits", "", "--abstain-instruction"])
        subprocess.run(["python", "scripts/label_grounding.py", "--run-dir", out_dir, "--labeler", LABELER], check=True)

if "relabel" in TASKS:
    for m in ("qwen", "phi"):
        for ds in GROUNDING_DATASETS:
            src = f"runs/phase3_grounding/{m}/{ds}"
            subprocess.run(["python", "scripts/label_grounding.py", "--run-dir", src, "--labeler", LABELER], check=True)
            dst = f"{OUT}/phase3_relabel/{m}/{ds}"
            os.makedirs(dst, exist_ok=True)
            shutil.copy(f"{src}/grounding.jsonl", f"{dst}/grounding_{LABELER}.jsonl")


## Package Output


In [ ]:
for root, _, files in os.walk(OUT):
    if "records.jsonl" in files:
        with open(f"{root}/records.jsonl") as f:
            print(root, sum(1 for _ in f), "records")
zip_path = shutil.make_archive(f"/kaggle/working/phase4a_{MODEL_TAG}_{TAG}", "zip", OUT)
print(zip_path, os.path.getsize(zip_path) // 1_000_000, "MB")
